In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

In [3]:
from marine_qc import (
    do_aground_check,
    do_multiple_sequential_check,
    do_speed_check,
    do_sst_biased_check,
)

In [4]:
from data import get_buoy_data

# How to use quality control checks with sequential reports from buoys

We need some text!!!

In [5]:
data = get_buoy_data()
data

,buoy_id,date,lat,lon,sst
0,buoy_1,2003-12-01,0.0,0.0,22.0
1,buoy_1,2003-12-02,1.0,0.0,21.6
2,buoy_1,2003-12-03,2.0,0.0,21.2
3,buoy_1,2003-12-04,5.0,0.0,20.8
4,buoy_1,2003-12-05,8.0,0.0,20.4
5,buoy_1,2003-12-06,9.0,0.0,20.0
6,buoy_1,2003-12-07,10.0,0.0,21.6
7,buoy_1,2003-12-08,11.0,0.0,21.2
8,buoy_1,2003-12-09,12.0,0.0,20.8
9,buoy_1,2003-12-10,13.0,0.0,20.4


In [6]:
qc_speed = do_speed_check(
    data.lon,
    data.lat,
    data.date,
    speed_limit=2.5,
    min_win_period=0.5,
    max_win_period=1.0,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "qc_speed": qc_speed})

,date,lat,lon,qc_speed
0,2003-12-01,0.0,0.0,0.0
1,2003-12-02,1.0,0.0,0.0
2,2003-12-03,2.0,0.0,1.0
3,2003-12-04,5.0,0.0,1.0
4,2003-12-05,8.0,0.0,1.0
5,2003-12-06,9.0,0.0,0.0
6,2003-12-07,10.0,0.0,0.0
7,2003-12-08,11.0,0.0,0.0
8,2003-12-09,12.0,0.0,0.0
9,2003-12-10,13.0,0.0,0.0


In [7]:
qc_aground = do_aground_check(
    data.lon,
    data.lat,
    data.date,
    smooth_win=3,
    min_win_period=1,
    max_win_period=2,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "qc_aground": qc_aground})

,date,lat,lon,qc_aground
0,2003-12-01,0.0,0.0,0.0
1,2003-12-02,1.0,0.0,0.0
2,2003-12-03,2.0,0.0,0.0
3,2003-12-04,5.0,0.0,0.0
4,2003-12-05,8.0,0.0,0.0
5,2003-12-06,9.0,0.0,0.0
6,2003-12-07,10.0,0.0,0.0
7,2003-12-08,11.0,0.0,0.0
8,2003-12-09,12.0,0.0,0.0
9,2003-12-10,13.0,0.0,0.0


In [8]:
ostia = pd.Series([22.0, 21.6, 21.2, 20.8, 20.4, 20.0, 19.6, 19.2, 18.8, 18.4, 18.0, 17.6, 17.2, 16.8, 16.4, 16.0, 16.0, 16.0, 16.0])
qc_biased = do_sst_biased_check(
    data.lon,
    data.lat,
    data.date,
    data.sst,
    ostia=ostia,
    ice=[0.0] * 19,
    bgvar=[0.01] * 19,
    n_eval=9,
    bias_lim=0.5,
    drif_intra=1.0,
    drif_inter=0.29,
    err_std_n=3.0,
    n_bad=2,
    background_err_lim=0.3,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "sst": data.sst, "qc_biased": qc_biased})

,date,lat,lon,sst,qc_biased
0,2003-12-01,0.0,0.0,22.0,1.0
1,2003-12-02,1.0,0.0,21.6,1.0
2,2003-12-03,2.0,0.0,21.2,1.0
3,2003-12-04,5.0,0.0,20.8,1.0
4,2003-12-05,8.0,0.0,20.4,1.0
5,2003-12-06,9.0,0.0,20.0,1.0
6,2003-12-07,10.0,0.0,21.6,1.0
7,2003-12-08,11.0,0.0,21.2,1.0
8,2003-12-09,12.0,0.0,20.8,1.0
9,2003-12-10,13.0,0.0,20.4,1.0


In [9]:
qc_dict = {
    "speed_check": {
        "func": "do_speed_check",
        "names": {
            "lats": "lat",
            "lons": "lon",
            "dates": "date",
        },
        "arguments": {
            "speed_limit": 2.5,
            "min_win_period": 1,
            "max_win_period": 2,
        },
    },
    "aground_check": {
        "func": "do_aground_check",
        "names": {
            "lats": "lat",
            "lons": "lon",
            "dates": "date",
        },
        "arguments": {
            "smooth_win": 3,
            "min_win_period": 1,
            "max_win_period": 2,
        },
    },
}

In [10]:
qc_multi = do_multiple_sequential_check(
    data,
    qc_dict=qc_dict,
    groupby="buoy_id",
    return_method="failed",
)
qc_multi

NameError: Function 'do_speed_check' is not defined.